# 1. Travelling Salesman Problem with Pyomo

Lecture: *Week 9 - Travelling Salesman Problem*, the slides from *MILP* to *Pyomo Code*, and the *Tutorial* on slide 22. Slides 20 and 22 link to this notebook on the course site; it runs in the browser.

**The problem.** A salesman must visit every city exactly once and return to the start, travelling as short a distance as possible. The notebook writes this as a mixed-integer linear program (MILP) with the Miller–Tucker–Zemlin (MTZ) constraints, solves it with the HiGHS solver, and then tries a heuristic, a genetic algorithm, on a larger problem.

**How to run it.** Run the cells in order. The first cell installs Pyomo and the HiGHS solver (allow a minute the first time); the rest takes well under a minute.

**Expected results.** 4 cities: the tour 1 → 2 → 4 → 3 → 1, length **80**. 20 random cities: one tour of length **407.97**, and without the MTZ constraints 10 separate two-city loops. Genetic algorithm on 35 cities: a tour of length **599.24**; the shortest tour for those cities is 518.52.

In [ ]:
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

## The TSP model with the Miller–Tucker–Zemlin (MTZ) constraints

x[i, j] = 1 if the tour goes from city i to city j; u[i] records the order in which the cities are visited. Each city has one arc in and one arc out, and the MTZ constraints rule out subtours. The example has 4 cities; the optimal tour has length 80.

The cell builds the model in `create_tsp_model` (a comment above each constraint says what it does), solves it with HiGHS (`tee=True` prints the solver's log), then prints the whole model (`model.pprint()`), the values of the variables and constraints (`model.display()`) and the tour.

*Note.* The original Colab run (with the GLPK solver) printed the same tour in the other direction, 1 → 3 → 4 → 2 → 1: the distances are symmetric, so both have length 80. The `u` values are not 2, 3, 4: the model gives `u` no bounds, so only their order matters; they increase along the tour after city 1.

In [ ]:
from pyomo.environ import *

def create_tsp_model(num_cities, distance_matrix):
    model = ConcreteModel()

    # Set of nodes
    model.N = RangeSet(1, num_cities)

    # Binary decision variables: x[i,j] equals 1 if the TSP tour goes from node i to node j
    model.x = Var(model.N, model.N, within=Binary, initialize=0)

    # Continuous decision variables: u[i] gives the order of node i in the tour (no bounds, so only the order of the values matters)
    model.u = Var(RangeSet(2, num_cities), within=Reals)

    # Objective function: minimize total distance
    model.obj = Objective(expr=sum(distance_matrix[i, j] * model.x[i, j] for i in model.N for j in model.N if i != j), sense=minimize)

    # Constraints
    # each city i is left exactly once: the sum over j of x[i,j] is 1
    def exactly_one_outgoing_arc_rule(model, i):
        return sum(model.x[i, j] for j in model.N if j != i) == 1

    model.exactly_one_outgoing_arc = Constraint(model.N, rule=exactly_one_outgoing_arc_rule)

    # each city i is entered exactly once: the sum over j of x[j,i] is 1
    def exactly_one_incoming_arc_rule(model, i):
        return sum(model.x[j, i] for j in model.N if j != i) == 1

    model.exactly_one_incoming_arc = Constraint(model.N, rule=exactly_one_incoming_arc_rule)

    # no arc from a city to itself (x is indexed N x N, so x[i,i] exists)
    def no_self_arc_rule(model, i):
        return model.x[i, i] == 0
    model.no_self_arc = Constraint(model.N, rule=no_self_arc_rule)

    # MTZ, for cities i and j other than city 1: if x[i,j] = 1 the right-hand side is 0, so u[j] >= u[i] + 1
    # (j comes after i); if x[i,j] = 0 it is n - 1 and the constraint does not restrict anything
    def mtz_subtour_elimination_rule(model, i, j):
        if i != j and j != 1 and i != 1:
            return model.u[i] - model.u[j] + 1 <= (num_cities - 1)*(1 - model.x[i,j])
        else:
            return Constraint.Skip
    model.mtz_subtour_elimination = Constraint(model.N, model.N, rule=mtz_subtour_elimination_rule)

    return model

def solve_tsp(model, tee=False):
    solver = SolverFactory('appsi_highs')  # HiGHS runs in the browser (the original Colab code used GLPK)
    solver.solve(model, tee=tee)
    return model

def print_solution(model):
    for i in model.N:
        for j in model.N:
            if value(model.x[i, j]) > 0.5:  # arc chosen (the solver may return 0.9999999 rather than exactly 1)
              print(f"Node {i} -> Node {j}")
    print(f"Optimal distance: {value(model.obj)}")

# Example usage
num_cities = 4
distance_matrix = {
    (1, 2): 10, (1, 3): 15, (1, 4): 20,
    (2, 1): 10, (2, 3): 35, (2, 4): 25,
    (3, 1): 15, (3, 2): 35, (3, 4): 30,
    (4, 1): 20, (4, 2): 25, (4, 3): 30
}

model = create_tsp_model(num_cities, distance_matrix)
solve_tsp(model, tee=True)
model.pprint()
model.display()
print_solution(model)

## 20 random cities, with and without the MTZ constraints

`generate_random_points` places the cities at random in a 100 × 100 square (`np.random.seed(0)` makes them the same on every run), `compute_distance_matrix` stores the straight-line distance between each pair, and `plot_tsp_solution` draws every arc with x[i, j] = 1.

The first plot shows the cities, the second the optimal tour (length 407.97). The model is then solved again **without** the MTZ constraints: the third plot shows 10 separate subtours of two cities each (total length 373.11: shorter, but not a tour), which is why the constraints are needed. The second and third plots are both titled *TSP Solution*, but only the second is a tour. Larger `num_cities` values take much longer to solve in the browser.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_random_points(num_cities):
    np.random.seed(0)
    # Generate random coordinates for each city
    city_coordinates = np.random.rand(num_cities, 2) * 100  # Assuming a 100x100 grid for simplicity
    return city_coordinates

def compute_distance_matrix(city_coordinates):
    num_cities = len(city_coordinates)
    distance_matrix = {}

    for i in range(num_cities):
        for j in range(num_cities):
            if i != j:
                distance_matrix[(i+1, j+1)] = np.linalg.norm(city_coordinates[i] - city_coordinates[j])  # Euclidean distance

    return distance_matrix


def plot_cities(city_coordinates):
    plt.figure(figsize=(8, 8))
    plt.scatter(city_coordinates[:, 0], city_coordinates[:, 1], c='red', label='Cities')
    for i, (x, y) in enumerate(city_coordinates):
        plt.annotate(f"{i+1}", (x, y), textcoords="offset points", xytext=(0,10), ha='center')
    plt.title('Cities')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_tsp_solution(city_coordinates, model):
  plt.figure(figsize=(8, 8))
  plt.scatter(city_coordinates[:, 0], city_coordinates[:, 1], c='red', label='Cities')
  for i, (x, y) in enumerate(city_coordinates):
      plt.annotate(f"{i+1}", (x, y), textcoords="offset points", xytext=(0,10), ha='center')

  # Plot the edges representing the TSP tour
  for i in model.N:
      for j in model.N:
          if value(model.x[i, j]) > 0.5:  # arc chosen (the solver may return 0.9999999 rather than exactly 1)
              plt.plot([city_coordinates[i-1, 0], city_coordinates[j-1, 0]],
                        [city_coordinates[i-1, 1], city_coordinates[j-1, 1]], 'b-')

  plt.title('TSP Solution')
  plt.xlabel('X Coordinate')
  plt.ylabel('Y Coordinate')
  plt.grid(True)
  plt.legend()
  plt.show()

# Example usage:
num_cities = 20
city_coordinates = generate_random_points(num_cities)
distance_matrix = compute_distance_matrix(city_coordinates)
model = create_tsp_model(num_cities, distance_matrix)
solve_tsp(model)

plot_cities(city_coordinates)
plot_tsp_solution(city_coordinates, model)
model.mtz_subtour_elimination.deactivate()  # switch the MTZ constraints off and solve again
solve_tsp(model)
plot_tsp_solution(city_coordinates, model)



## A heuristic: genetic algorithm on 35 cities

A genetic algorithm searches for a good tour without proving it is optimal. A *population* of 100 random tours evolves for 200 *generations*: *tournament selection* picks parents (the shortest of 5 random tours), *ordered crossover* combines two parents into two children that are still valid tours, *swap mutation* exchanges two cities in a child, and the best 100 tours among parents and children survive. Its "fitness" is the tour length, so smaller is better.

It makes random choices; the cell fixes the seed (`random.seed(1)`), so every run gives the same tour, of length 599.24. Change or delete the seed to see how much the result varies. The MILP finds the shortest tour for these 35 cities, 518.52, in about 20 seconds in the browser: the genetic algorithm's tour is 15.6 % longer.

*Tutorial (slide 22): increase the problem size and see when the MILP becomes too slow to solve; change the population, crossover and mutation settings; can the genetic algorithm handle larger problems than the MILP?*

In [ ]:
import numpy as np
import random

def generate_random_population(num_individuals, num_cities):
    return [random.sample(range(1, num_cities + 1), num_cities) for _ in range(num_individuals)]

def compute_fitness(individual, distance_matrix):
    # fitness = length of the tour (smaller is better)
    fitness = 0
    num_cities = len(individual)
    for i in range(num_cities - 1):
        fitness += distance_matrix[(individual[i], individual[i+1])]
    fitness += distance_matrix[(individual[-1], individual[0])]  # Complete the tour
    return fitness

def tournament_selection(population, distance_matrix, tournament_size):
    selected_parents = []
    for _ in range(len(population)):
        tournament = random.sample(population, tournament_size)
        selected_parents.append(min(tournament, key=lambda x: compute_fitness(x, distance_matrix)))
    return selected_parents

def ordered_crossover(parent1, parent2):
    size = len(parent1)
    offspring1, offspring2 = [-1] * size, [-1] * size
    start, end = sorted(random.sample(range(size), 2))
    offspring1[start:end] = parent1[start:end]
    offspring2[start:end] = parent2[start:end]

    p1_indices, p2_indices = set(parent1[start:end]), set(parent2[start:end])
    p1_remaining, p2_remaining = [x for x in parent2 if x not in p1_indices], [x for x in parent1 if x not in p2_indices]

    # fill the other places with the missing cities in the other parent's order; the slice assignments resize
    # the list, so a child is one parent's segment followed by those cities (still a valid tour)
    offspring1[end:] = [x for x in p1_remaining if x not in offspring1]
    offspring1[:start] = [x for x in p1_remaining if x not in offspring1]

    offspring2[end:] = [x for x in p2_remaining if x not in offspring2]
    offspring2[:start] = [x for x in p2_remaining if x not in offspring2]

    return offspring1, offspring2

def swap_mutation(individual):
    idx1, idx2 = random.sample(range(len(individual)), 2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual

def genetic_algorithm(num_cities, distance_matrix, population_size, num_generations, tournament_size, crossover_rate, mutation_rate):
    population = generate_random_population(population_size, num_cities)
    for _ in range(num_generations):
        parents = tournament_selection(population, distance_matrix, tournament_size)
        offspring = []
        for i in range(0, len(parents), 2):
            parent1 = parents[i]
            parent2 = parents[i+1]
            if random.random() < crossover_rate:
                child1, child2 = ordered_crossover(parent1, parent2)
            else:
                child1, child2 = parent1[:], parent2[:]
            if random.random() < mutation_rate:
                child1 = swap_mutation(child1)
            if random.random() < mutation_rate:
                child2 = swap_mutation(child2)
            offspring.append(child1)
            offspring.append(child2)
        # keep the best population_size tours among parents and children
        population = sorted(population + offspring, key=lambda x: compute_fitness(x, distance_matrix))[:population_size]
    best_solution = min(population, key=lambda x: compute_fitness(x, distance_matrix))
    best_fitness = compute_fitness(best_solution, distance_matrix)
    return best_solution, best_fitness

def plot_tsp_solution_ga(city_coordinates, best_solution):
    plt.figure(figsize=(8, 8))
    plt.scatter(city_coordinates[:, 0], city_coordinates[:, 1], c='red', label='Cities')
    for i, (x, y) in enumerate(city_coordinates):
        plt.annotate(f"{i+1}", (x, y), textcoords="offset points", xytext=(0,10), ha='center')

    # Plot the edges representing the TSP tour
    for i in range(len(best_solution)):
        city1 = best_solution[i]
        city2 = best_solution[(i + 1) % len(best_solution)]  # Connect back to the first city
        plt.plot([city_coordinates[city1-1, 0], city_coordinates[city2-1, 0]],
                 [city_coordinates[city1-1, 1], city_coordinates[city2-1, 1]], 'b-')

    plt.title('TSP Solution')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.grid(True)
    plt.legend()
    plt.show()

# Example usage:
num_cities = 35
city_coordinates = generate_random_points(num_cities)
distance_matrix = compute_distance_matrix(city_coordinates)
population_size = 100
num_generations = 200
tournament_size = 5
crossover_rate = 0.7
mutation_rate = 0.2

random.seed(1)  # fixed seed: every run gives the same tour; change or delete it to see other runs
best_solution, best_fitness = genetic_algorithm(num_cities, distance_matrix, population_size, num_generations, tournament_size, crossover_rate, mutation_rate)

print("Best solution:", best_solution)
print("Best fitness:", best_fitness)

# Example usage:
plot_tsp_solution_ga(city_coordinates, best_solution)


## What do you notice?

Compare the second and third plots of the 20-city cell. Without the MTZ constraints, why does every city end up in a pair?

<details><summary>Answer</summary>

Without MTZ the model only asks for one arc into and one arc out of every city (the *Subtour example* slide). The cheapest way to do that is to go from each city to a nearby city and straight back: 10 two-city loops with total length 373.11. Every city has one arc in and one out, yet there is no single tour. The MTZ constraints make u increase along every arc that does not touch city 1, so a loop that misses city 1 is impossible, and the result is one tour of length 407.97.
</details>

## Try it

In the 20-city cell, change `num_cities = 20` to `30` and run that cell again. How long does the first solve take, and how long is the tour? Then, in the last code cell, change `mutation_rate = 0.2` to `0.8`. Does the genetic algorithm get closer to the shortest tour, 518.52?

<details><summary>Answer</summary>

30 cities take about 6 seconds in the browser (about a second on a desktop); the shortest tour has length 511.32. The solve time grows quickly but unevenly with the number of cities: 35 cities take about 20 seconds and 40 cities about two and a half minutes. With `mutation_rate = 0.8` the genetic algorithm finds 573.02 (10.5 % above the shortest tour) instead of 599.24 (15.6 %).
</details>